# The mass law in the grid world

Settles the findings of `sigreg_hypothesis.ipynb` and `grid_benchmark.ipynb` in one controlled world. The working rule: **a content
class is held in a latent of size D only if its share m of the pixel variance satisfies m · D ≥ κ** (κ ≈ 1.5). The share is the variance
over clips that the class contributes to the frames: a component that looks the same in every clip contributes nothing.

**Settings for every run.** SIGReg weight 0.01 (live encoder and shadow alike); random map; frame 48 × 48: a 12 × 8 map and 4 HUD rows
(16 inventory slots, up to 16 hand tiles, up to 16 compass tiles); one seed per cell, the last cell repeats steps 1, 4, 5 and 7 for two more
seeds (optional). Probes: a standardised MLP per part (inventory, position, facing, hand, mob, twinkle) of the latent and, for B, of its
shadow, plus the effective rank; read once at the end of every run. Kept = MLP R² ≥ 0.5, lost = R² < 0.2.

**The models**
- `detached`: one encoder trained by SIGReg alone (a predictor trains on its detached latents and is not read). Availability from mass alone.
- `ae`: the same encoder plus a decoder, trained by pixel reconstruction and SIGReg, no prediction. This is exactly how B's shadow encoder
  is trained (its predictor sees detached latents), here on its own.
- step 4 only: `sg` (prediction + SIGReg), A = `target` with a head start of 3000 steps, B = `retain_open`.

| step | question | runs |
|---|---|---|
| 0 | the mass of every component, the threshold κ/D per D | none (CPU) |
| 1 | does what is held follow the hand's mass? (hand shown in k = 1, 2, 4, 8, 16 tiles, D = 32) | 10 |
| 2 | does the threshold fall as 1/D? (the same at D = 16, 64, 128) | 30 |
| 3 | does frequency count like size? (equal mass: 4 tiles always, 8 tiles in half of the clips, 16 in a quarter) | 4 |
| 5 | is location the barrier for the facing? (the facing in a fixed compass of c = 2, 4, 8 tiles; c = 2 has the marker's mass) | 6 |
| 4 | the recipe: below and above the threshold (hand k = 2 and 16), `sg`, A and B, 20 000 steps | 6 |
| 7 | class-balanced reconstruction: the `ae` with each tile's error weighted by its ground-truth class's share^−p (p = 0.5, 1) | 4 |

**Predictions, written before any run**
- **1:** the hand's readout rises with its mass and crosses 0.5 near κ/32 = 4.7% (between k = 4 and 8).
- **2:** the crossing mass falls about as 1/D: the slope of log(crossing) against log(D) lies between −1.3 and −0.7. Both models.
- **3:** the three equal-mass settings agree within 0.15 (readout on the clips where the hand is shown).
- **5:** the compass is held from about the threshold, as the hand; the moving marker is not held at the same mass (c = 2).
- **4:** below the threshold all three lose the hand. Above it `sg` loses it, A keeps it, B keeps it and drops the noise (mob, twinkle ≤ 0.3).
  If A is within 0.15 of B, mass replaces most of the shadow.
- **7:** facing and hand reach 0.5 at the default masses; the noise readouts rise with them (the risk of the weighting).

Run order: 0, 1, 2, 3, 5, 4, 7. Budget (estimate from earlier A40 timings, 3 workers, the larger frame): steps 1–2 about an hour, 3, 5 and 7
about 20 minutes together, step 4 about an hour. Every run is resumable: finished runs in `OUT` are skipped.

In [ ]:
%pip install -q -e .
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from gaf.env_grid import GridCfg, component_masses
from gaf.train import Cfg
from gaf import analysis as A

OUT = "/workspace/results_masslaw"
SEED = 0
EXTRA_SEEDS = (1, 2)                  # the optional cell at the end
WORKERS = 3                           # runs at once on the GPU
KAPPA = 1.5

# ---- the knobs ---------------------------------------------------------------------------
FRAME = dict(W=12, H=8)               # 48 x 48 pixels: a 12 x 8 map and 4 HUD rows
K = (1, 2, 4, 8, 16)                  # tiles that show the hand (steps 1, 2)
DS = (16, 32, 64, 128)                # latent sizes (step 2; 32 is step 1)
PRESENCE = ((8, 0.5), (16, 0.25))     # step 3: (tiles, share of the clips), each with the mass of 4 tiles always shown
COMPASS = (2, 4, 8)                   # step 5: compass tiles
RECIPE_K = (2, 16)                    # step 4: below and above the threshold at D = 32
HEAD_START = 3000                     # step 4: variant A
CB_POWERS, CB_K = (0.5, 1.0), (1, 2)  # step 7
PARTS = ["hand", "facing", "mob", "twinkle", "inventory", "position"]

LEAN = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, steps=5000, floor_start=10 ** 9, eval_every=1000)   # no gate: no floor probes
FULL = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, steps=20_000, basis="excess", floor_steps=1500, floor_warm=True)
pd.set_option("display.precision", 3)


def grid(**kw):
    return GridCfg(**{**FRAME, **kw})


def lean(arm, env, tag, D=32, seed=SEED, **kw):
    return Cfg(arm=arm, seed=seed, D=D, env=env, tag=tag, **{**LEAN, **kw})


def full(arm, env, tag, seed=SEED, **kw):
    return Cfg(arm=arm, seed=seed, D=32, env=env, tag=tag, **{**FULL, **kw})


def pick(df, seeds=(SEED,), **sel):
    d = A.select(df, **sel)
    return d[d.seed.isin(seeds)]


def crossing(masses, readouts, thr=0.5):
    """The mass at which the readout reaches thr, log-linear between the swept points; (value, note)."""
    m, r = np.asarray(masses, float), np.asarray(readouts, float)
    hit = np.where(r >= thr)[0]
    if not len(hit):
        return np.nan, f"> {m[-1]:.2f}%"
    i = hit[0]
    if i == 0:
        return np.nan, f"< {m[0]:.2f}%"
    f = (thr - r[i - 1]) / (r[i] - r[i - 1])
    v = float(np.exp(np.log(m[i - 1]) + f * (np.log(m[i]) - np.log(m[i - 1]))))
    return v, f"{v:.2f}%"

## 0 The masses
Each component's share of the pixel variance in this frame (2048 clips, CPU, about a minute), the hand at every k, the compass at every c,
and the threshold κ/D. The predicted crossing: the first k whose mass reaches κ/D.

In [ ]:
base = component_masses(grid(), n=2048)
display(pd.Series({k: round(100 * v, 2) for k, v in base.items()}, name="% of the pixel variance").to_frame())
MASS = {k: 100 * component_masses(grid(hand_tiles=k), n=2048)["hand"] for k in K}
CMASS = {c: 100 * component_masses(grid(compass_tiles=c), n=2048)["compass"] for c in COMPASS}
PMASS = {(k, p): 100 * component_masses(grid(hand_tiles=k, hand_p=p), n=2048)["hand"] for k, p in PRESENCE}
MARKER = 100 * base["facing marker"]
display(pd.DataFrame({"hand mass %": MASS}).T.round(2))
display(pd.DataFrame({"compass mass %": CMASS}).T.round(2))
display(pd.DataFrame({"hand mass % (with presence)": {f"k={k}, p={p}": v for (k, p), v in PMASS.items()}}).T.round(2))
pred = {}
for D in DS:
    tau = 100 * KAPPA / D
    first = next((k for k in K if MASS[k] >= tau), None)
    pred[D] = {"threshold %": round(tau, 2), "first k at or above": first}
display(pd.DataFrame(pred).T)

## 1 The hand's mass at D = 32
`detached` and `ae`, hand shown in k = 1, 2, 4, 8, 16 tiles, 5000 steps. The table gives every part's readout (MLP, standardised) and the
effective rank; the facing marker, mob and twinkle do not change with k and show what else each model holds.

In [ ]:
E1 = [lean(arm, grid(hand_tiles=k), f"k{k}") for arm in ("detached", "ae") for k in K]
A.execute(E1, OUT, workers=WORKERS)
df = A.collect(OUT)


def sweep(df, arm, D, seeds=(SEED,)):
    rows = {}
    for k in K:
        r = pick(df, seeds, arm=arm, tag=f"k{k}", D=D)
        if len(r):
            rows[f"k={k} ({MASS[k]:.2f}%)"] = r[[f"pm_{p}" for p in PARTS] + ["pm_rank"]].mean()
    return pd.DataFrame(rows).T


for arm in ("detached", "ae"):
    print(arm, "D = 32")
    display(sweep(df, arm, 32).round(3))

## 2 The threshold against D
The same sweep at D = 16, 64 and 128 (30 runs). For every model and D: the mass at which the hand's readout crosses 0.5, and the slope of
log(crossing) against log(D) (the rule: −1).

In [ ]:
E2 = [lean(arm, grid(hand_tiles=k), f"k{k}", D=D) for arm in ("detached", "ae") for D in DS if D != 32 for k in K]
A.execute(E2, OUT, workers=WORKERS)
df = A.collect(OUT)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True, layout="constrained")
CROSS = {}
for ax, arm in zip(axes, ("detached", "ae")):
    for D, color in zip(DS, A.SERIES):
        rows = [(MASS[k], pick(df, arm=arm, tag=f"k{k}", D=D)["pm_hand"].mean()) for k in K]
        rows = [(m, r) for m, r in rows if not np.isnan(r)]
        if not rows:
            continue
        m, r = zip(*rows)
        ax.plot(m, r, marker="o", color=color, label=f"D={D}")
        ax.axvline(100 * KAPPA / D, color=color, ls="--", lw=1)
        CROSS[(arm, D)] = crossing(m, r)
    ax.set(xscale="log", title=arm, xlabel="hand's share of the pixel variance (%)", ylim=(-0.05, 1.02))
axes[0].set_ylabel("MLP readout of the hand")
axes[1].legend(fontsize=8)
t = pd.DataFrame({arm: {D: CROSS.get((arm, D), (np.nan, "-"))[1] for D in DS} for arm in ("detached", "ae")})
t["predicted (kappa/D)"] = [f"{100 * KAPPA / D:.2f}%" for D in DS]
display(t)
SLOPE = {}
for arm in ("detached", "ae"):
    pts = [(D, CROSS[(arm, D)][0]) for D in DS if (arm, D) in CROSS and not np.isnan(CROSS[(arm, D)][0])]
    SLOPE[arm] = np.polyfit(np.log([p[0] for p in pts]), np.log([p[1] for p in pts]), 1)[0] if len(pts) >= 2 else np.nan
    print(f"{arm}: slope of log(crossing) against log(D) = {SLOPE[arm]:.2f} from {len(pts)} bracketed crossings (rule: -1)")

**Reading 1–2.** A rising readout with a crossing near κ/D and a slope near −1: the rule holds for discrete, fixed-location content, and
κ can be read off (crossing × D). A crossing that does not move with D: the limit is not the latent's size, and the rule's D is the wrong variable.
`ae` crossing at a lower mass than `detached`: reconstruction lowers the threshold (it adds a reason to keep content beyond its mass).

## 3 Frequency against size
The hand shown in 8 tiles in half of the clips and in 16 tiles in a quarter of them, both with the mass of 4 tiles always shown (step 1's
k = 4 run). Hidden tiles show the average hand colour, as an absent block in Craftax shows ordinary terrain. The readout is taken on the
clips where the hand is shown (`pm_hand_visible`).

In [ ]:
E3 = [lean(arm, grid(hand_tiles=k, hand_p=p), f"p{p}-k{k}") for arm in ("detached", "ae") for k, p in PRESENCE]
A.execute(E3, OUT, workers=WORKERS)
df = A.collect(OUT)
rows = {}
for arm in ("detached", "ae"):
    r = pick(df, arm=arm, tag="k4", D=32)
    if len(r):
        rows[(arm, f"4 tiles, always ({MASS[4]:.2f}%)")] = {"hand readout": r["pm_hand"].mean(), "rank": r["pm_rank"].mean()}
    for k, p in PRESENCE:
        r = pick(df, arm=arm, tag=f"p{p}-k{k}", D=32)
        if len(r):
            rows[(arm, f"{k} tiles, {int(100 * p)}% of clips ({PMASS[(k, p)]:.2f}%)")] = {
                "hand readout": r["pm_hand_visible"].mean(), "rank": r["pm_rank"].mean()}
T3 = pd.DataFrame(rows).T
display(T3.round(3))

## 5 The facing in a fixed place
The facing shown in a compass of c = 2, 4, 8 HUD tiles (c = 2 has about the marker's mass), `detached` and `ae`, D = 32. Compared with the
marker alone (step 1, k = 1: no compass) and with the hand at similar masses (step 1).

In [ ]:
E5 = [lean(arm, grid(compass_tiles=c), f"c{c}") for arm in ("detached", "ae") for c in COMPASS]
A.execute(E5, OUT, workers=WORKERS)
df = A.collect(OUT)
rows = {}
for arm in ("detached", "ae"):
    r = pick(df, arm=arm, tag="k1", D=32)
    if len(r):
        rows[(arm, f"marker only ({MARKER:.2f}%)")] = {"facing readout": r["pm_facing"].mean()}
    for c in COMPASS:
        r = pick(df, arm=arm, tag=f"c{c}", D=32)
        if len(r):
            rows[(arm, f"compass c={c} ({CMASS[c]:.2f}%)")] = {"facing readout": r["pm_facing"].mean()}
    for k in (2, 4, 8):
        r = pick(df, arm=arm, tag=f"k{k}", D=32)
        if len(r):
            rows[(arm, f"hand k={k} ({MASS[k]:.2f}%)")] = {"facing readout": np.nan, "hand readout": r["pm_hand"].mean()}
T5 = pd.DataFrame(rows).T
display(T5.round(3))

**Reading 3 and 5.** Step 3: equal readouts mean frequency counts like size, so a rare class can be lifted by sampling; a lower readout for
the rare settings means rarity costs more than its mass says. Step 5: a compass held like the hand at the same mass while the marker is not
means the moving location is the barrier for the facing, not its mass.

## 4 The recipe
The hand in k = 2 tiles (below the threshold at D = 32) and in 16 tiles (above it). `sg`, A = `target` with a head start of 3000 steps (the
encoder sees only SIGReg while the predictor learns), B = `retain_open`. 20 000 steps, with the floor gate on. The table reads the live
encoder (the model that predicts) and, for B, its shadow; the settledness check looks at the last 4000 steps.

In [ ]:
E4 = [full(arm, grid(hand_tiles=k), f"recipe-k{k}", head_start=HEAD_START if arm == "target" else 0)
      for k in RECIPE_K for arm in ("sg", "target", "retain_open")]
A.execute(E4, OUT, workers=WORKERS)
df = A.collect(OUT)
NAME = {"sg": "sg", "target": "A: target + head start", "retain_open": "B: retain_open"}
rows, runs = {}, {}
for k in RECIPE_K:
    for arm in ("sg", "target", "retain_open"):
        r = pick(df, arm=arm, tag=f"recipe-k{k}")
        if len(r):
            row = {f"live {p}": r[f"pm_{p}"].mean() for p in PARTS}
            if arm == "retain_open":
                row.update({f"shadow {p}": r[f"sh_pm_{p}"].mean() for p in ("hand", "facing", "mob")})
            rows[(f"k={k} ({MASS[k]:.2f}%)", NAME[arm])] = row
            runs[f"k={k} {NAME[arm]}"] = dict(arm=arm, tag=f"recipe-k{k}", seed=SEED)
T4 = pd.DataFrame(rows).T
display(T4.round(3))
print("settled? change and swing over the last 4000 steps (ridge readouts of the eval rows)")
display(A.drift(df, OUT, runs, cols=("r2_hard", "r2_fast", "r2_noise", "r2_slow")))

**Reading 4.** Above the threshold: `sg` low and A and B high means the content is available and only the race removes it; A within 0.15 of
B means a head start does the shadow's job there, and B is needed only where SIGReg alone does not hold the content or where noise must go
(compare the mob and twinkle columns). B high below the threshold as well would contradict the rule. Unsettled runs are provisional.

## 7 Class-balanced reconstruction
The `ae` with each tile's reconstruction error weighted by its ground-truth tile class's share of the variance to the power −p (p = 0.5
and 1; capped at 30×, mean weight 1), at the default masses (hand k = 1 and 2). Compared with the plain `ae` of step 1. The noise columns
show the risk: rare noise tiles (the mob) are lifted too.

In [ ]:
E7 = [lean("ae", grid(hand_tiles=k), f"cb{pw}-k{k}", rec_class_power=pw) for pw in CB_POWERS for k in CB_K]
A.execute(E7, OUT, workers=WORKERS)
df = A.collect(OUT)
rows = {}
for k in CB_K:
    for pw, tag in [(0.0, f"k{k}")] + [(pw, f"cb{pw}-k{k}") for pw in CB_POWERS]:
        r = pick(df, arm="ae", tag=tag, D=32)
        if len(r):
            rows[(f"k={k} ({MASS[k]:.2f}%)", f"power {pw}")] = {**{p: r[f"pm_{p}"].mean() for p in PARTS},
                                                               "rank": r["pm_rank"].mean(), "rec mse": r["ae_rec_mse"].mean()}
T7 = pd.DataFrame(rows).T
display(T7.round(3))

## Verdicts
The rules above, applied to whatever has run.

In [ ]:
def say(label, ok):
    print(f"{label}: {'holds' if ok else 'does not hold'}")


try:
    s1 = {arm: sweep(df, arm, 32)["pm_hand"] for arm in ("detached", "ae")}
    for arm, s in s1.items():
        print(f"step 1, {arm}: hand readout by mass", s.round(2).to_dict())
        say(f"  rises with mass ({arm})", s.iloc[-1] - s.iloc[0] >= 0.3)
except Exception as e:  # noqa: BLE001
    print("step 1: not run yet", e)
if "SLOPE" in globals():
    for arm, v in SLOPE.items():
        say(f"step 2, {arm}: slope {v:.2f} within [-1.3, -0.7]", -1.3 <= v <= -0.7)
if "T3" in globals():
    for arm in ("detached", "ae"):
        v = T3.loc[arm, "hand readout"]
        say(f"step 3, {arm}: equal-mass settings within 0.15 (spread {v.max() - v.min():.2f})", v.max() - v.min() <= 0.15)
if "T5" in globals():
    for arm in ("detached", "ae"):
        t = T5.loc[arm]
        m = t.loc[t.index.str.startswith("marker"), "facing readout"]
        c2 = t.loc[t.index.str.startswith("compass c=2"), "facing readout"]
        if len(m) and len(c2):
            say(f"step 5, {arm}: compass c=2 ({c2.iloc[0]:.2f}) above the marker ({m.iloc[0]:.2f}) by 0.3", c2.iloc[0] - m.iloc[0] >= 0.3)
if "T4" in globals():
    hi, lo = f"k={RECIPE_K[1]} ({MASS[RECIPE_K[1]]:.2f}%)", f"k={RECIPE_K[0]} ({MASS[RECIPE_K[0]]:.2f}%)"
    try:
        h = T4.loc[hi, "live hand"]
        say("step 4, above: sg loses the hand (< 0.2)", h[NAME["sg"]] < 0.2)
        say("step 4, above: A keeps it (>= 0.5)", h[NAME["target"]] >= 0.5)
        say("step 4, above: B keeps it (>= 0.5)", h[NAME["retain_open"]] >= 0.5)
        say("step 4, above: B drops the noise (mob, twinkle <= 0.3)",
            max(T4.loc[(hi, NAME["retain_open"]), "live mob"], T4.loc[(hi, NAME["retain_open"]), "live twinkle"]) <= 0.3)
        say("step 4, above: A within 0.15 of B", abs(h[NAME["target"]] - h[NAME["retain_open"]]) <= 0.15)
        say("step 4, below: all three lose it (< 0.2)", (T4.loc[lo, "live hand"] < 0.2).all())
    except KeyError as e:
        print("step 4: not complete", e)
if "T7" in globals():
    for k in CB_K:
        t = T7.loc[f"k={k} ({MASS[k]:.2f}%)"]
        best = t.drop("power 0.0", errors="ignore")
        say(f"step 7, k={k}: facing and hand reach 0.5 with the weighting", ((best["facing"] >= 0.5) & (best["hand"] >= 0.5)).any())

## All seeds (optional)
Steps 1, 4, 5 and 7 again for `EXTRA_SEEDS`, then the seed means and spreads of the hand (and facing) readouts. About 2.5 times the time of
those steps.

In [ ]:
RUN_ALL_SEEDS = False
if RUN_ALL_SEEDS:
    extra = []
    for s in EXTRA_SEEDS:
        extra += [lean(arm, grid(hand_tiles=k), f"k{k}", seed=s) for arm in ("detached", "ae") for k in K]
        extra += [lean(arm, grid(compass_tiles=c), f"c{c}", seed=s) for arm in ("detached", "ae") for c in COMPASS]
        extra += [full(arm, grid(hand_tiles=k), f"recipe-k{k}", seed=s, head_start=HEAD_START if arm == "target" else 0)
                  for k in RECIPE_K for arm in ("sg", "target", "retain_open")]
        extra += [lean("ae", grid(hand_tiles=k), f"cb{pw}-k{k}", seed=s, rec_class_power=pw) for pw in CB_POWERS for k in CB_K]
    A.execute(extra, OUT, workers=WORKERS)
    df = A.collect(OUT)
    seeds = (SEED, *EXTRA_SEEDS)
    d = df[df.seed.isin(seeds) & (df.D == 32)]
    for col in ("pm_hand", "pm_facing"):
        print(col, "(mean, sd over seeds)")
        display(d.pivot_table(index="tag", columns="arm", values=col, aggfunc=["mean", "std"]).round(3))

## All runs

In [ ]:
df = A.collect(OUT)
df.to_csv(f"{OUT}/summary.csv", index=False)
print(len(df), "runs;", f"{df.seconds.sum() / 3600:.1f} GPU-hours")